In [4]:
!pip install langchain langchain-community langgraph pypdf

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 1.4 MB/s eta 0:00:02
   -------- ------------------------------- 0.5/2.5 MB 1.4 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.5 MB 877.4 kB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.5 MB 877.4 kB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.5 MB 877.4 kB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.5 MB 877.4 kB/s eta 0:00:02
   ---------------- ----------------------- 1.0/2.5 MB 573.3 kB/s eta 0:00:03
   ---------------- ----------------------- 1.0/2.5 MB 573.3 kB/s eta 0:00:03
   ---------------- ----------------------- 1.0/2.5 MB 573.3 kB/s eta 0:00:03
   -------------------- ------------------- 1.3/2.5 MB 533.2 kB/s eta 0:00:03
   -------------------- ------------------- 1.3/2.5 MB 533.2 kB/s eta 0:00:03
   -----


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\santh\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langgraph.graph import StateGraph

# -----------------------------
# LOAD PDF
# -----------------------------
loader = PyPDFLoader("support.pdf")
documents = loader.load()

# -----------------------------
# LINE-BASED CHUNKING
# -----------------------------
chunks = []
for doc in documents:
    lines = doc.page_content.split("\n")
    for line in lines:
        if line.strip():
            chunks.append(line.strip())

# -----------------------------
# SMART RETRIEVAL (UPDATED)
# -----------------------------
def retrieve_chunks(query):
    query = query.lower()

    for i, chunk in enumerate(chunks):
        text = chunk.lower()

        if "overview" in query and "overview" in text:
            return [chunk]

        if "feature" in query and "feature" in text:
            return [chunk + " " + (chunks[i+1] if i+1 < len(chunks) else "")]

        if "workflow" in query and "workflow" in text:
            return [chunk + " " + (chunks[i+1] if i+1 < len(chunks) else "")]

        if "technology" in query and ("technology" in text or "langchain" in text):
            return [chunk]

        if "benefit" in query and "benefit" in text:
            return [chunk + " " + (chunks[i+1] if i+1 < len(chunks) else "")]

        if "limitation" in query and "limitation" in text:
            return [chunk + " " + (chunks[i+1] if i+1 < len(chunks) else "")]

        if "support" in query and "support" in text:
            return [chunk + " " + (chunks[i+1] if i+1 < len(chunks) else "")]

        if "email" in query and "email" in text:
            return [chunk]

        if "phone" in query and "phone" in text:
            return [chunk]

    return []

# -----------------------------
# GENERATE ANSWER
# -----------------------------
def generate_answer(query):
    docs = retrieve_chunks(query)

    if not docs:
        return "NO_CONTEXT"

    return docs[0]

# -----------------------------
# CONFIDENCE CHECK
# -----------------------------
def check_confidence(answer):
    if answer == "NO_CONTEXT":
        return "HUMAN"
    return "OK"

# -----------------------------
# GRAPH NODES
# -----------------------------
def process_node(state):
    answer = generate_answer(state["query"])
    return {"query": state["query"], "answer": answer}

def decision_node(state):
    decision = check_confidence(state["answer"])
    return {**state, "decision": decision}

def human_node(state):
    print("\n⚠️ Escalated to Human Support")
    human = input("Enter manual response: ")
    return {**state, "answer": human}

def output_node(state):
    return state

# -----------------------------
# GRAPH WORKFLOW
# -----------------------------
graph = StateGraph(dict)

graph.add_node("process", process_node)
graph.add_node("decision", decision_node)
graph.add_node("human", human_node)
graph.add_node("output", output_node)

graph.set_entry_point("process")
graph.add_edge("process", "decision")

def route(state):
    return "human" if state["decision"] == "HUMAN" else "output"

graph.add_conditional_edges("decision", route)
graph.add_edge("human", "output")

app = graph.compile()

# -----------------------------
# RUN LOOP
# -----------------------------
while True:
    query = input("\nAsk your question (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    result = app.invoke({"query": query})
    print("\nAnswer:", result["answer"])


Ask your question (type 'exit' to quit):  what is overview?



Answer: 1. Overview



Ask your question (type 'exit' to quit):  tell me features



Answer: 2. Key Features  Extract skills, tools, and experience  Match candidates with job descriptions  Assign fit score



Ask your question (type 'exit' to quit):  what is workflow?



Answer: 3. Workflow Resume → Skill Extraction → Matching → Scoring → Explanation
